In [1]:
import polars as pl
from pathlib import Path
from Bio import SeqIO

In [2]:
genome_source = Path("../igannot/sandbox/new_atlas").resolve()
genome_source

PosixPath('/data/users/bdupin/igannot/sandbox/new_atlas')

In [3]:
CWD = Path().resolve()
BASE = CWD

print(f"Folder where groups (data) folder should be: {BASE}")

Folder where groups (data) folder should be: /data/users/bdupin/sam-honours


In [4]:
group = {
    "plants_plt_solanaceae": "plants_plt"
}

gene_pairs = [

]

In [6]:
group_ans_by_gp = {}
for g, gsource in group.items():
    print(f"Processing group: {g} from source: {gsource}")
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")

    filtered_gp = pl.read_csv(BASE / g / f"gene_pairs_filtered.tsv", separator="\t").select(["gene_pair", "igr_count"])

    atlas = pl.read_csv(BASE / g / f"{g}_summary_atlas.tsv", separator="\t")

    if not gene_pairs:
        gene_pairs = filtered_gp.select("gene_pair").to_series().to_list()

    grouped = (
    atlas.filter(pl.col("gene_pair").is_in(gene_pairs))
         .group_by(["gene_pair", "AN"])
         .agg(pl.col("ID"))
        )


    ans_by_gp = {}
    for gp, an, ids in grouped.iter_rows():
        ans_by_gp.setdefault(gp, {})[an] = ids
    
    group_ans_by_gp[g] = ans_by_gp
    

Processing group: plants_plt_solanaceae from source: plants_plt


In [ ]:
for g, gsource in group.items():
    for gp, ans_dict in group_ans_by_gp[g].items():
        print(f"Processing gene pair: {gp}")
        records = []
        for an, ids in ans_dict.items():
            fasta_file = genome_source / gsource / an /f"{an}_intergenic.fasta"
            if not fasta_file.exists():
                print(f"    Fasta file {fasta_file} does not exist. Skipping.")
            
            output_dir = BASE / g / "gene_pair_fastas"
            output_dir.mkdir(parents=True, exist_ok=True)

            for seq_record in SeqIO.parse(fasta_file, "fasta"):
                if seq_record.id in ids:
                    records.append(seq_record)

        SeqIO.write(records, output_dir / f"gp_{gp}.fasta", "fasta")

Processing gene pair: TH|PSBA|--
Processing gene pair: YCF2|YCF15|++
Processing gene pair: NDHB|TL1|2|++
Processing gene pair: PSBB|PSBT|++
Processing gene pair: PSBJ|PSBF|--
Processing gene pair: PETN|PSBM|+-
Processing gene pair: TM|RPS14|--
Processing gene pair: TL1|2|NDHB|--
Processing gene pair: PSAI|YCF4|++
Processing gene pair: RPS7|NDHB|++
Processing gene pair: NDHB|RPS7|--
Processing gene pair: RNR23|TA|--
Processing gene pair: PSBK|PSBI|++
Processing gene pair: TM|ATPE|+-
Processing gene pair: ATPF|ATPH|--
Processing gene pair: YCF2|TL1|2|+-
Processing gene pair: NDHH|RPS15|--
Processing gene pair: TT|PSBD|++
Processing gene pair: RPS15|YCF1|--
Processing gene pair: ATPB|RBCL|-+
Processing gene pair: TT|TL1|2|-+
Processing gene pair: PSBH|PETB|++
Processing gene pair: NDHC|TV|--
Processing gene pair: PSBE|PETL|-+
Processing gene pair: TV|RNRS|++
Processing gene pair: TA|RNR23|++
Processing gene pair: RNR16|TI|++
Processing gene pair: RPS2|RPOC2|--
Processing gene pair: NDHE|N